# AI-Powered Supply Chain Attack Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import networkx as nx
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import hashlib
import re
from collections import defaultdict, deque
import ast

class SupplyChainAttackDetector:
    def __init__(self):
        self.dependency_graph = nx.DiGraph()
        self.malware_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        self.anomaly_detector = IsolationForest(contamination=0.05, random_state=42)
        self.code_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1,3))
        self.scaler = StandardScaler()
        self.trust_scores = defaultdict(float)

    def analyze_package_dependencies(self, packages):
        for pkg in packages:
            self.dependency_graph.add_node(pkg['name'], **pkg)
            for dep in pkg.get('dependencies', []):
                self.dependency_graph.add_edge(pkg['name'], dep['name'], version=dep.get('version', '*'))
        return self.detect_suspicious_dependency_patterns()

    def detect_suspicious_dependency_patterns(self):
        suspicious_packages = []
        for node in self.dependency_graph.nodes():
            in_degree = self.dependency_graph.in_degree(node)
            out_degree = self.dependency_graph.out_degree(node)
            node_data = self.dependency_graph.nodes[node]

            risk_score = 0
            if node_data.get('downloads', 0) < 100 and out_degree > 5: risk_score += 0.3
            if re.search(r'[0-9]{4,}|temp|test|fake', node.lower()): risk_score += 0.2
            if node_data.get('author_trust', 1.0) < 0.3: risk_score += 0.4
            if in_degree == 0 and out_degree > 10: risk_score += 0.3

            if risk_score > 0.5:
                suspicious_packages.append({'package': node, 'risk_score': risk_score})
        return suspicious_packages

    def analyze_code_changes(self, code_diffs):
        malicious_patterns = [
            r'eval\s*\(', r'exec\s*\(', r'__import__\s*\(', r'subprocess\.', r'os\.system',
            r'urllib\.request', r'requests\.get.*?http', r'base64\.decode', r'socket\.socket'
        ]

        features = []
        for diff in code_diffs:
            added_lines = [line for line in diff.split('\n') if line.startswith('+')]
            removed_lines = [line for line in diff.split('\n') if line.startswith('-')]

            malicious_score = sum(len(re.findall(pattern, '\n'.join(added_lines))) for pattern in malicious_patterns)
            entropy = self.calculate_entropy('\n'.join(added_lines))
            obfuscation_score = self.detect_obfuscation(added_lines)
            size_ratio = len(added_lines) / max(len(removed_lines), 1)

            features.append([malicious_score, entropy, obfuscation_score, size_ratio, len(added_lines)])
        return np.array(features)

    def calculate_entropy(self, text):
        if not text: return 0
        char_counts = defaultdict(int)
        for char in text:
            char_counts[char] += 1
        probs = np.array(list(char_counts.values())) / len(text)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def detect_obfuscation(self, code_lines):
        obfuscation_indicators = 0
        for line in code_lines:
            if len(re.findall(r'[a-zA-Z_][a-zA-Z0-9_]{20,}', line)) > 2: obfuscation_indicators += 1
            if line.count('\\x') > 3: obfuscation_indicators += 1
            if re.search(r'chr\(\d+\)', line): obfuscation_indicators += 1
        return obfuscation_indicators / max(len(code_lines), 1)

    def build_trust_network(self, maintainers, packages):
        trust_graph = nx.Graph()
        for maintainer in maintainers:
            trust_graph.add_node(maintainer['id'], **maintainer)

        for pkg in packages:
            for maintainer_id in pkg.get('maintainers', []):
                if maintainer_id in trust_graph:
                    trust_graph.add_edge(pkg['name'], maintainer_id)

        centrality = nx.betweenness_centrality(trust_graph)
        for node, score in centrality.items():
            self.trust_scores[node] = score
        return self.identify_compromised_accounts(trust_graph)

    def identify_compromised_accounts(self, trust_graph):
        compromised_candidates = []
        for node in trust_graph.nodes():
            node_data = trust_graph.nodes.get(node, {})
            suspicious_score = 0

            if node_data.get('recent_activity_spike', False): suspicious_score += 0.3
            if node_data.get('location_changes', 0) > 2: suspicious_score += 0.2
            if node_data.get('failed_logins', 0) > 10: suspicious_score += 0.4
            if len(list(trust_graph.neighbors(node))) > 50: suspicious_score += 0.1

            if suspicious_score > 0.4:
                compromised_candidates.append({'account': node, 'risk': suspicious_score})
        return compromised_candidates

    def detect_supply_chain_attack(self, packages, code_diffs, maintainers):
        suspicious_deps = self.analyze_package_dependencies(packages)
        code_features = self.analyze_code_changes(code_diffs)
        compromised_accounts = self.build_trust_network(maintainers, packages)

        if len(code_features) > 0:
            anomalies = self.anomaly_detector.fit_predict(self.scaler.fit_transform(code_features))
            code_anomaly_ratio = np.mean(anomalies == -1)
        else:
            code_anomaly_ratio = 0

        attack_score = len(suspicious_deps) * 0.3 + code_anomaly_ratio * 0.4 + len(compromised_accounts) * 0.3
        return attack_score > 0.5, {'deps': suspicious_deps, 'accounts': compromised_accounts, 'score': attack_score}